# 🌿 Actividad Final — Clasificación de Enfermedades en Plantas con CNN
### Dataset: PlantVillage · Transfer Learning · Ejecución 100% local

**Curso:** PFAD-PADCC-VIC04 · Visión por Computadora · TecNM Virtual 2025

---

## ¿Qué aprenderás con este notebook?

Entrenas una CNN para **detectar automáticamente enfermedades en hojas de plantas**
a partir de fotografías. Este es un problema real de **visión agrícola por IA**:
sistemas similares se usan en campo para alertar a agricultores antes de que
una plaga se extienda y destruya una cosecha completa.

---

## El dataset PlantVillage

PlantVillage contiene **54,306 imágenes** de hojas de 14 especies de plantas,
organizadas en **38 clases** que combinan planta + enfermedad (o planta + sana).

Ejemplos de clases:
```
Apple___Apple_scab           ← manzano con sarna
Apple___healthy              ← manzano sano
Corn___Common_rust           ← maíz con roya común
Potato___Late_blight         ← papa con tizón tardío (causó la Gran Hambruna Irlandesa)
Tomato___Tomato_mosaic_virus ← tomate con virus del mosaico
```

Las clases siguen el patrón `Planta___Condicion`. Hay 38 combinaciones posibles.

---

## Descarga del dataset

1. Ve a: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset. Descarga el ZIP (~1 GB)
3. Pon la ruta en `ZIP_PATH` de la **Celda 0**

El ZIP de Kaggle contiene:
```
PlantVillage.zip
└── PlantVillage/
    ├── color/       ← imágenes RGB (las que usamos)
    ├── grayscale/   ← ignorar
    └── segmented/   ← ignorar
```

---

## Archivos que genera este notebook

```
proyecto_plantas/
├── data/
│   ├── distribucion.png        ← barras con conteo por clase
│   ├── muestras_dataset.png    ← muestras de clases (máx 10)
│   ├── curvas.png              ← loss / accuracy / F1 por época
│   ├── confusion_counts.png    ← matriz de confusión (conteos)
│   ├── confusion_norm.png      ← matriz de confusión (recall por clase)
│   ├── metricas_clase.png      ← precision/recall/F1 por clase
│   ├── predicciones_grid.png   ← 16 predicciones del test set
│   ├── historial.csv           ← métricas por época
│   └── reporte.csv             ← reporte de clasificación
└── models/
    ├── modelo_best.pth         ← mejor checkpoint
    ├── modelo_produccion.pth   ← listo para FastAPI / despliegue
    └── idx2clase.json          ← mapeo 0→Apple___scab, 1→..., etc.
```

---

## Tiempo estimado de entrenamiento

| Hardware | `PC_DEBIL` | Tiempo por época | Total estimado |
|----------|-----------|-----------------|----------------|
| CPU básica (< 8 GB RAM) | `True` | 10–20 min | 2–4 h |
| CPU media (≥ 8 GB RAM) | `False` | 20–40 min | 7–13 h |
| GPU dedicada | `False` | 1–4 min | 20–80 min |

> PlantVillage tiene ~54k imágenes (5× más que HAM10000), por eso el entrenamiento
> tarda más. Con `PC_DEBIL = True` es mucho más manejable.

> **Alta accuracy esperada:** las clases tienen diferencias visuales muy marcadas
> (manchas de colores distintos, texturas únicas). Con EfficientNet-B2 se puede
> llegar a >92% de accuracy en el test set.


---
## Celda 0 — Configuración
> **Edita solo esta celda.** El resto del notebook corre automáticamente.

In [ ]:
# =============================================================
# ⚙️  CONFIGURACIÓN — edita solo aquí
# =============================================================

# ─── RUTA DEL ZIP ─────────────────────────────────────────────
# Descarga el ZIP de Kaggle y escribe aquí la ruta completa.
# Ejemplos:
#   Windows : r'C:\Users\maria\Downloads\plantdisease.zip'
#   Mac/Linux: '/home/maria/Downloads/plantdisease.zip'
ZIP_PATH = r'C:\Users\usuario\Downloads\plantdisease.zip'

# ─── CARPETA DE SALIDA ────────────────────────────────────────
BASE_PATH_STR = r'C:\Users\usuario\proyecto_plantas'

# ─── MODO DE HARDWARE ─────────────────────────────────────────
# True  → MobileNetV3-Large, batch=8, imagen 128×128, 10 épocas
#         Recomendado si tu PC tiene < 8 GB de RAM o es lenta
# False → EfficientNet-B2, batch=16, imagen 224×224, 20 épocas
#         Recomendado si tienes ≥ 8 GB de RAM o GPU
PC_DEBIL = False

# ─── HIPERPARÁMETROS AVANZADOS (deja en None para automático) ─
MODEL_OVERRIDE  = None
BATCH_OVERRIDE  = None
IMG_OVERRIDE    = None
EPOCHS_OVERRIDE = None

# ─── OTROS ───────────────────────────────────────────────────
LR        = 3e-4
PATIENCE  = 5
SEED      = 42
VAL_SIZE  = 0.15
TEST_SIZE = 0.15

# =============================================================
print('✅ Configuración cargada')
print(f'   ZIP      : {ZIP_PATH}')
print(f'   Salida   : {BASE_PATH_STR}')
print(f'   PC débil : {PC_DEBIL}')


---
## Celda 1 — Instalación de dependencias

| Paquete | Para qué sirve |
|---------|---------------|
| `torch` + `torchvision` | Framework de deep learning |
| `timm` | Modelos preentrenados (EfficientNet-B2, MobileNetV3…) |
| `albumentations` | Augmentación de imágenes de hojas |
| `torchmetrics` | Accuracy y F1-score por época |
| `scikit-learn` | Matriz de confusión y reporte detallado |
| `seaborn` | Heatmaps de la matriz de confusión |
| `pandas` | Manejo de CSV e historial de entrenamiento |


In [ ]:
import subprocess, sys

paquetes = ['torch','torchvision','timm','albumentations',
            'torchmetrics','scikit-learn','seaborn','pandas','Pillow']

print('Instalando dependencias...')
for pkg in paquetes:
    r = subprocess.run([sys.executable,'-m','pip','install','-q',pkg], capture_output=True)
    print(f'  {"✅" if r.returncode==0 else "❌"} {pkg}')
print('\n✅ Listo. Continúa con Celda 2.')


---
## Celda 2 — Imports y configuración del entorno

Esta celda:
- Importa todas las bibliotecas necesarias
- Detecta si hay GPU disponible
- Define hiperparámetros según `PC_DEBIL`
- Crea las carpetas del proyecto


In [ ]:
import os, json, shutil, zipfile, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
matplotlib.use('Agg')  # guarda figuras como PNG sin abrir ventanas
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchmetrics import Accuracy, F1Score
from sklearn.metrics import classification_report, confusion_matrix
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# Semilla global — reproducibilidad
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# Dispositivo
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Dispositivo : {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('   CPU mode')
    print(f'   PC_DEBIL = {PC_DEBIL} — {"modo ligero activo ✅" if PC_DEBIL else "Tip: cambia a True si el entrenamiento es muy lento"}')

# Hiperparámetros según hardware
if PC_DEBIL:
    _model, _batch, _img, _epochs = 'mobilenetv3_large_100', 8,  128, 10
else:
    _model, _batch, _img, _epochs = 'efficientnet_b2',       16, 224, 20

MODEL_NAME = MODEL_OVERRIDE  or _model
BATCH_SIZE = BATCH_OVERRIDE  or _batch
IMG_SIZE   = IMG_OVERRIDE    or _img
EPOCHS     = EPOCHS_OVERRIDE or _epochs
NUM_WORKERS = 0   # siempre 0 en local (evita errores de multiprocessing en Windows)

# Estadísticas de ImageNet — necesarias porque el backbone fue preentrenado con ellas
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Carpetas del proyecto
BASE_PATH   = Path(BASE_PATH_STR)
IMAGES_PATH = BASE_PATH / 'images'
ZIP_TMP     = BASE_PATH / 'tmp_zip'
for d in ['data','images','models','tmp_zip']:
    (BASE_PATH / d).mkdir(parents=True, exist_ok=True)

print(f'\n📂 Proyecto en: {BASE_PATH}')
print(f'\n⚙️  Hiperparámetros:')
print(f'   Modelo    : {MODEL_NAME}')
print(f'   Batch     : {BATCH_SIZE}')
print(f'   Img size  : {IMG_SIZE}×{IMG_SIZE} px')
print(f'   Épocas    : {EPOCHS}')


---
## Celda 3 — Descompresión del ZIP

PlantVillage viene con esta estructura interna:
```
plantdisease.zip
└── PlantVillage/
    ├── color/
    │   ├── Apple___Apple_scab/
    │   │   ├── 0a61f2f9-9012-4ce9-bae0-3c8a5572babb.jpg
    │   │   └── ...
    │   ├── Apple___healthy/
    │   └── ... (38 carpetas en total)
    ├── grayscale/   ← se ignora automáticamente
    └── segmented/   ← se ignora automáticamente
```

La Celda 4 detecta automáticamente la subcarpeta `color/` y la usa.


In [ ]:
zip_path = Path(ZIP_PATH)
if not zip_path.exists():
    raise FileNotFoundError(
        f'\n❌ ZIP no encontrado: {zip_path}'
        f'\n   Verifica ZIP_PATH en Celda 0.'
        f'\n   Descarga en: https://www.kaggle.com/datasets/emmarex/plantdisease'
    )

tam_mb = zip_path.stat().st_size / 1e6
print(f'📦 Descomprimiendo {zip_path.name} ({tam_mb:.0f} MB)...')
print('   Esto puede tardar 3–7 minutos.')

with zipfile.ZipFile(zip_path, 'r') as zf:
    print(f'   Archivos en el ZIP: {len(zf.namelist()):,}')
    zf.extractall(str(ZIP_TMP))

print(f'✅ Descompresión completada → {ZIP_TMP}')


---
## Celda 4 — Organización del dataset en train / val / test

PlantVillage viene con 38 carpetas de clases (dentro de `color/`) pero
**sin división** en entrenamiento, validación y test.

Esta celda:
1. Detecta automáticamente la subcarpeta `color/` e ignora `grayscale/` y `segmented/`
2. Encuentra las 38 carpetas de clases dentro de `color/`
3. Divide cada clase en **70% train / 15% val / 15% test** de forma estratificada
4. Copia cada imagen a `images/split/clase/imagen.jpg`

La división es **estratificada**: el mismo porcentaje de cada clase va a cada split,
lo que garantiza que los tres conjuntos sean representativos.


In [ ]:
# =============================================================
# 🔍 DETECCIÓN AUTOMÁTICA DE LA CARPETA color/
# PlantVillage tiene color/, grayscale/ y segmented/.
# Solo usamos color/ — imágenes RGB de mayor calidad diagnóstica.
# =============================================================
def ignorar_carpeta(nombre):
    """Carpetas/archivos que no son datos de imágenes."""
    return (nombre.startswith('__') or nombre.startswith('.')
            or nombre.lower() in {'grayscale','segmented','gray','__macosx'}
            or nombre.lower().endswith(('.csv','.txt','.json','.md','.py')))

def imgs_en(carpeta):
    """Lista de imágenes en una carpeta (no recursivo)."""
    out = []
    for ext in ['*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG']:
        out.extend(carpeta.glob(ext))
    return out

def encontrar_color(base):
    """
    Navega recursivamente hasta encontrar la carpeta 'color/'.
    Maneja estructuras anidadas como:
      PlantVillage/ → color/ → clases/
      plantdisease/ → PlantVillage/ → color/ → clases/
    """
    carpetas = [p for p in base.iterdir()
                if p.is_dir() and not ignorar_carpeta(p.name)]
    nombres  = {p.name.lower(): p for p in carpetas}

    if 'color' in nombres:
        print(f'   ✅ Carpeta color/ encontrada: {nombres["color"]}')
        return nombres['color']

    # No encontramos color/ aquí → bajar un nivel si hay una sola carpeta
    if len(carpetas) == 1:
        print(f'   Bajando un nivel: {carpetas[0].name}/')
        return encontrar_color(carpetas[0])

    # Múltiples carpetas sin color/ → buscar en cada una
    for c in carpetas:
        resultado = encontrar_color(c)
        if resultado is not None:
            return resultado

    return None


# ── Buscar color/ ──────────────────────────────────────────────
print('🔍 Buscando carpeta color/ dentro del ZIP...')
color_dir = encontrar_color(ZIP_TMP)

if color_dir is None:
    raise ValueError(
        '❌ No se encontró la carpeta color/ en el ZIP.\n'
        '   Asegúrate de descargar el ZIP correcto de PlantVillage en Kaggle.\n'
        '   Ejecuta la Celda 4b para ver la estructura del ZIP.'
    )

# Listar las 38 clases dentro de color/
clases_dirs = sorted([p for p in color_dir.iterdir()
                       if p.is_dir() and not ignorar_carpeta(p.name)])
print(f'   Clases encontradas: {len(clases_dirs)}')

# =============================================================
# 📁 SPLIT ESTRATIFICADO 70/15/15
# =============================================================
TRAIN_DIR = IMAGES_PATH / 'train'
VAL_DIR   = IMAGES_PATH / 'val'
TEST_DIR  = IMAGES_PATH / 'test'

print(f'\n📁 Dividiendo en train/val/test (70/15/15)...')
resumen = {}
for clase_dir in clases_dirs:
    imgs = imgs_en(clase_dir)
    if not imgs:
        print(f'   ⚠️  {clase_dir.name} — sin imágenes, omitiendo')
        continue

    random.seed(SEED)
    random.shuffle(imgs)

    n     = len(imgs)
    n_val = max(1, int(n * VAL_SIZE))
    n_tst = max(1, int(n * TEST_SIZE))
    n_tr  = n - n_val - n_tst

    for sname, simgs in [('train', imgs[:n_tr]),
                          ('val',   imgs[n_tr:n_tr+n_val]),
                          ('test',  imgs[n_tr+n_val:])]:
        dst = IMAGES_PATH / sname / clase_dir.name
        dst.mkdir(parents=True, exist_ok=True)
        for img in simgs:
            shutil.copy2(str(img), str(dst / img.name))

    resumen[clase_dir.name] = n
    print(f'   {clase_dir.name:<45} total={n:>4}  train={n_tr:>3}  val={n_val:>3}  test={n_tst:>3}')

# Limpiar temporal
shutil.rmtree(str(ZIP_TMP), ignore_errors=True)
print('\n🗑️  Carpeta temporal eliminada')

# Verificación
print('\n✅ Verificación final:')
total = 0
for split in ['train','val','test']:
    sp = IMAGES_PATH / split
    clases_sp = [p for p in sp.iterdir() if p.is_dir()]
    n = sum(len(imgs_en(c)) for c in clases_sp)
    total += n
    print(f'   {split:<6}: {len(clases_sp):>2} clases · {n:>7,} imágenes')
print(f'   Total : {total:,} imágenes')
print('\n🟢 Dataset listo. Continúa con Celda 5.')


---
## Celda 4b — Diagnóstico (solo si Celda 4 falló)

In [ ]:
def arbol(path, nivel=0, max_n=3, max_i=10):
    if nivel > max_n: return
    try:
        items = sorted(path.iterdir())[:max_i]
    except Exception: return
    for item in items:
        pre = '  ' * nivel
        if item.is_dir():
            n = len(list(item.rglob('*.jpg')))
            print(f'{pre}📂 {item.name}/  ({n} imgs)')
            arbol(item, nivel+1, max_n, max_i)
        elif item.suffix.lower() in ['.csv','.txt','.json']:
            print(f'{pre}📄 {item.name}')

if not ZIP_TMP.exists() or not any(ZIP_TMP.iterdir()):
    ZIP_TMP.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(Path(ZIP_PATH),'r') as zf:
        zf.extractall(str(ZIP_TMP))

print('Árbol del ZIP:')
print('='*55)
arbol(ZIP_TMP)


---
## Celda 5 — Análisis visual del dataset

PlantVillage tiene una distribución **relativamente balanceada** comparada con HAM10000:
la mayoría de las clases tiene entre 300 y 2,000 imágenes.

Las clases están nombradas con el patrón `Planta___Condicion`, donde la condición
puede ser el nombre de una enfermedad o simplemente `healthy` (sana).

Nota cómo las imágenes de hojas sanas tienen un verde uniforme y sin manchas,
mientras que las enfermas muestran manchas de colores característicos de cada patología.


In [ ]:
# =============================================================
# 📊 INVENTARIO DE CLASES
# =============================================================
clases_train = sorted([p.name for p in (IMAGES_PATH/'train').iterdir() if p.is_dir()])
NUM_CLASES   = len(clases_train)
clase2idx    = {c: i for i, c in enumerate(clases_train)}
idx2clase    = {str(i): c for c, i in clase2idx.items()}

print(f'Clases detectadas: {NUM_CLASES}')

conteos = {}
for split in ['train','val','test']:
    sp = IMAGES_PATH / split
    if not sp.exists(): continue
    for cd in sp.iterdir():
        if not cd.is_dir(): continue
        n = len(imgs_en(cd))
        conteos.setdefault(cd.name, {})[split] = n

df_dist = pd.DataFrame(conteos).T.fillna(0).astype(int)
df_dist['total'] = df_dist.sum(axis=1)
df_dist = df_dist.sort_values('total', ascending=False)
print(f'\nTop 10 clases por tamaño:')
print(df_dist.head(10).to_string())

with open(BASE_PATH/'data'/'idx2clase.json','w',encoding='utf-8') as f:
    json.dump(idx2clase, f, ensure_ascii=False, indent=2)
print(f'\n✅ idx2clase.json guardado ({NUM_CLASES} clases)')


In [ ]:
# =============================================================
# 📊 GRÁFICA DE DISTRIBUCIÓN — 38 clases
# Con tantas clases la gráfica es alta: se guarda como PNG.
# =============================================================
altura = max(8, NUM_CLASES * 0.32)
fig, axes = plt.subplots(1, 2, figsize=(16, altura))
fig.suptitle('PlantVillage — Distribución de clases (38 tipos)',
             fontsize=13, fontweight='bold')

colores = ['#e74c3c' if v < 100 else '#f39c12' if v < 500 else '#27ae60'
           for v in df_dist['total'].values]
axes[0].barh(df_dist.index, df_dist['total'], color=colores)
axes[0].set_xlabel('Imágenes totales')
axes[0].set_title('Distribución por clase')
axes[0].tick_params(axis='y', labelsize=7)
for i, v in enumerate(df_dist['total']):
    axes[0].text(v+5, i, str(v), va='center', fontsize=6)
parches = [mpatches.Patch(color='#e74c3c',label='<100'),
           mpatches.Patch(color='#f39c12',label='100-500'),
           mpatches.Patch(color='#27ae60',label='>500')]
axes[0].legend(handles=parches, fontsize=8)

splits_tot = {s: df_dist[s].sum() for s in ['train','val','test'] if s in df_dist.columns}
axes[1].pie(splits_tot.values(), labels=splits_tot.keys(),
            autopct='%1.1f%%', colors=['#3498db','#e67e22','#2ecc71'],
            startangle=90, textprops={'fontsize':11})
axes[1].set_title('Proporción train / val / test')

plt.tight_layout()
out = BASE_PATH / 'data' / 'distribucion.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'📁 distribucion.png guardado → {out}')
print('   (Abre el PNG para ver la distribución completa de las 38 clases)')


In [ ]:
# =============================================================
# 🖼️  MUESTRAS — máx 10 clases, 3 imágenes por clase
# Se muestran 10 clases al azar para no generar una figura enorme.
# =============================================================
n_vis = min(10, NUM_CLASES)
vis_c = random.sample(clases_train, n_vis)
COLS  = 3

fig, axes = plt.subplots(n_vis, COLS, figsize=(COLS * 3.5, n_vis * 3.5))
if n_vis == 1: axes = [axes]
fig.suptitle(f'PlantVillage — Muestras de {n_vis} clases (hojas sanas y enfermas)',
             fontsize=12, fontweight='bold')

for fila, clase in enumerate(vis_c):
    imgs = imgs_en(IMAGES_PATH / 'train' / clase)
    random.shuffle(imgs)
    for col in range(COLS):
        ax = axes[fila][col] if n_vis > 1 else axes[col]
        if col < len(imgs):
            try:
                ax.imshow(Image.open(imgs[col]).convert('RGB'))
            except Exception: pass
        ax.axis('off')
        if col == 0:
            # Mostrar solo la parte después de ___ para que quepa
            partes = clase.split('___')
            etiq   = f'{partes[0]}\n{partes[1]}' if len(partes)>1 else clase
            ax.set_ylabel(etiq, fontsize=7, rotation=0, labelpad=75, va='center')

plt.tight_layout()
out = BASE_PATH / 'data' / 'muestras_dataset.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'📁 muestras_dataset.png guardado → {out}')


---
## Celda 6 — Augmentación y DataLoaders

### ¿Por qué augmentación especial para hojas de plantas?

Las imágenes de PlantVillage se tomaron en condiciones controladas
(fondo uniforme, iluminación constante). En campo real, las hojas
aparecen en cualquier orientación, con diferentes iluminaciones y
parcialmente ocluidas por otras hojas.

La augmentación simula esas condiciones reales:

| Transformación | Propósito |
|---------------|-----------|
| `HorizontalFlip` | Una hoja enferma lo es independientemente del lado |
| `RandomRotate90` | La orientación de la hoja no define la enfermedad |
| `ShiftScaleRotate` | Simula que la hoja no está centrada |
| `ColorJitter` | Simula variaciones de luz solar |
| `CLAHE` | Mejora contraste → manchas más visibles |
| `RandomBrightnessContrast` | Simula distintos momentos del día |
| `GaussNoise` | Simula ruido de la cámara del celular en campo |

### WeightedRandomSampler

PlantVillage está bastante balanceado pero tiene algunas clases con más
imágenes (Tomato tiene muchas más variantes que Blueberry). El sampler
garantiza que el modelo vea todas las clases con frecuencia similar.


In [ ]:
# =============================================================
# 🎨 AUGMENTACIÓN PARA IMÁGENES DE HOJAS
# =============================================================
if PC_DEBIL:
    train_aug = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, p=0.4),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])
    print('[PC_DEBIL] Augmentación ligera')
else:
    train_aug = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.12,
                           rotate_limit=20, p=0.5),
        A.ColorJitter(brightness=0.25, contrast=0.25,
                      saturation=0.2, hue=0.08, p=0.5),
        A.CLAHE(clip_limit=2.5, p=0.35),            # realza manchas de enfermedades
        A.RandomBrightnessContrast(p=0.3),           # variaciones de iluminación solar
        A.GaussianBlur(blur_limit=(3,5), p=0.2),
        A.GaussNoise(var_limit=(5,20), p=0.2),
        A.CoarseDropout(max_holes=3, max_height=20,  # simula oclusión por otras hojas
                        max_width=20, p=0.2),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])
    print('[Normal] Augmentación completa para imágenes de hojas')

# Sin augmentación en validación/test: evaluamos las imágenes tal como son
val_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

# =============================================================
# 📂 DATASET PERSONALIZADO
# =============================================================
class PlantDataset(Dataset):
    """
    Lee imágenes de images/split/clase/imagen.jpg
    y aplica las transformaciones de albumentations.
    Maneja imágenes corruptas devolviendo un tensor negro.
    """
    def __init__(self, split_path, clase2idx, transform=None):
        self.samples   = []
        self.transform = transform
        for clase_dir in sorted(split_path.iterdir()):
            if not clase_dir.is_dir(): continue
            idx = clase2idx.get(clase_dir.name)
            if idx is None: continue
            for ext in ['*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG']:
                for img_path in clase_dir.glob(ext):
                    self.samples.append((img_path, idx))

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        try:
            img = np.array(Image.open(path).convert('RGB'))
        except Exception:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(label, dtype=torch.long)


train_ds = PlantDataset(IMAGES_PATH/'train', clase2idx, train_aug)
val_ds   = PlantDataset(IMAGES_PATH/'val',   clase2idx, val_aug)
test_ds  = PlantDataset(IMAGES_PATH/'test',  clase2idx, val_aug)

# WeightedRandomSampler — balancea clases con distinto número de imágenes
etiquetas_train = [s[1] for s in train_ds.samples]
conteo_clases   = [etiquetas_train.count(i) for i in range(NUM_CLASES)]
pesos_muestras  = [1.0 / max(conteo_clases[lbl], 1) for lbl in etiquetas_train]
sampler         = WeightedRandomSampler(pesos_muestras, len(train_ds), replacement=True)

# NUM_WORKERS=0 siempre en local (evita errores de multiprocessing en Windows/Mac)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, pin_memory=(DEVICE=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE=='cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE=='cuda'))

print('✅ DataLoaders listos')
print(f'   train : {len(train_ds):,} imágenes · {len(train_loader)} batches')
print(f'   val   : {len(val_ds):,} imágenes · {len(val_loader)} batches')
print(f'   test  : {len(test_ds):,} imágenes · {len(test_loader)} batches')


---
## Celda 7 — Modelo CNN con Transfer Learning

### ¿Por qué EfficientNet-B2 para PlantVillage?

PlantVillage tiene **38 clases** y ~54k imágenes. EfficientNet-B2 está bien
calibrado para este rango: no es tan grande como para ser lento en CPU,
pero tiene suficiente capacidad para distinguir 38 patrones visuales distintos.

La accuracy esperada es alta (>90%) porque las diferencias entre clases son
visualmente muy marcadas: cada enfermedad tiene manchas, texturas y colores
característicos que la CNN aprende a reconocer después de ver miles de ejemplos.

### ¿Qué aprenden las capas?

- **Capas iniciales:** bordes y texturas básicas (como los filtros Sobel del Tema 2)
- **Capas medias:** manchas circulares, patrones de decoloración, texturas rugosas
- **Capas finales:** la combinación específica que define cada enfermedad

En la Celda 10 visualizaremos los filtros aprendidos por la primera capa.


In [ ]:
# =============================================================
# 🧠 MODELO — EfficientNet-B2 / MobileNetV3 para PlantVillage (38 clases)
# =============================================================
class ClasificadorPlantas(nn.Module):
    """
    CNN para clasificación de enfermedades en plantas (PlantVillage, 38 clases).

    Usa un backbone preentrenado en ImageNet como extractor de características.
    Las capas del backbone ya saben detectar bordes, texturas y formas.
    Solo entrenamos la cabeza de clasificación, que aprende qué combinación
    de características corresponde a cada enfermedad.

    Cabeza:
      Dropout(0.5) → evita que el modelo dependa de pocas características
      Linear(feats→512) → combina características en representación compacta
      GELU() → activación no lineal
      Dropout(0.4) → más regularización
      Linear(512→38) → una salida por clase (logit)
    """
    def __init__(self, num_classes, model_name=MODEL_NAME, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0, global_pool='avg',
        )
        in_f = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_f, min(512, in_f)),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(min(512, in_f), num_classes),
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))


model     = ClasificadorPlantas(NUM_CLASES).to(DEVICE)
total_p   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅ Modelo: {MODEL_NAME}')
print(f'   Parámetros totales     : {total_p/1e6:.1f} M')
print(f'   Parámetros entrenables : {trainable/1e6:.1f} M')
print(f'   Clases de salida       : {NUM_CLASES}')

with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out   = model(dummy)
    print(f'   Forward pass OK        : {list(dummy.shape)} → {list(out.shape)}')
del dummy, out


---
## Celda 8 — Entrenamiento

### Notas específicas para PlantVillage

**38 clases requieren más capacidad de discriminación.** El modelo necesita
aprender patrones muy específicos para cada combinación planta-enfermedad.
Por eso usamos EfficientNet-B2 (más profundo) en lugar de MobileNet.

**El dataset está más balanceado que HAM10000**, así que los pesos de clase
son menos extremos. Sin embargo, algunas plantas tienen más enfermedades
documentadas que otras, por lo que el balanceo sigue siendo útil.

**Accuracy esperada:** con EfficientNet-B2 y 20 épocas, se puede llegar a
**92–97%** de accuracy en el test set. PlantVillage es un dataset "amigable"
porque las clases son visualmente muy distintas entre sí.


In [ ]:
# =============================================================
# 🚀 ENTRENAMIENTO — PlantVillage (38 clases)
# =============================================================

# Pesos de clase: inversamente proporcionales a la frecuencia
cc = [etiquetas_train.count(i) for i in range(NUM_CLASES)]
cls_weights = torch.tensor(
    [len(etiquetas_train) / (NUM_CLASES * max(c, 1)) for c in cc],
    dtype=torch.float32,
).to(DEVICE)

# CrossEntropyLoss con peso de clase y label smoothing
# label_smoothing=0.1 → las etiquetas duras (0/1) se suavizan a (0.05/0.95)
# Evita que el modelo sea excesivamente confiado → mejor generalización
criterion = nn.CrossEntropyLoss(weight=cls_weights, label_smoothing=0.1)

# AdamW: Adam con weight decay correcto (regularización L2 en pesos, no en bias)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)

# CosineAnnealingWarmRestarts: el LR baja en coseno y se reinicia en T_0=10 épocas
# Permite escapar de mínimos locales y afinar la convergencia
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

acc_m = Accuracy(task='multiclass', num_classes=NUM_CLASES).to(DEVICE)
f1_m  = F1Score(task='multiclass', num_classes=NUM_CLASES, average='macro').to(DEVICE)

MODEL_SAVE = BASE_PATH / 'models' / 'modelo_best.pth'


def train_epoch():
    """Una época: forward → loss → backward → actualizar pesos."""
    model.train()
    total_loss = 0.0
    acc_m.reset()
    for imgs, labels in train_loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        # Clip de gradientes: evita la explosión del gradiente
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        acc_m.update(logits.detach(), labels)
    return total_loss / len(train_loader), acc_m.compute().item()


@torch.no_grad()
def evaluate(loader):
    """Evaluación sin gradientes → más rápido, sin consumir memoria de gradientes."""
    model.eval()
    total_loss = 0.0
    acc_m.reset()
    f1_m.reset()
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item()
        acc_m.update(logits, labels)
        f1_m.update(logits, labels)
    return total_loss / len(loader), acc_m.compute().item(), f1_m.compute().item()


# ── Bucle principal ────────────────────────────────────────────
best_acc, no_imp, history = 0.0, 0, []
sep = '─' * 82
print(f'🚀 Iniciando entrenamiento — PlantVillage')
print(f'   Modelo  : {MODEL_NAME} | Clases: {NUM_CLASES} | Device: {DEVICE}')
print(f'   Épocas  : {EPOCHS} máx | Patience: {PATIENCE}')
print(sep)
print(f'  {"Epoch":>5} | {"TrainLoss":>9} | {"TrainAcc":>8} | '
      f'{"ValLoss":>8} | {"ValAcc":>7} | {"ValF1":>7}')
print(sep)

for epoch in range(1, EPOCHS + 1):
    tl, ta      = train_epoch()
    vl, va, vf1 = evaluate(val_loader)
    scheduler.step(epoch)

    history.append({'epoch':epoch,'train_loss':tl,'train_acc':ta,
                    'val_loss':vl,'val_acc':va,'val_f1':vf1})

    mejoro = va > best_acc
    print(f'  {epoch:5d} | {tl:9.4f} | {ta:8.4f} | '
          f'{vl:8.4f} | {va:7.4f} | {vf1:7.4f}  {"🟢" if mejoro else ""}')

    if mejoro:
        best_acc = va; no_imp = 0
        torch.save({
            'epoch':epoch,'model_state_dict':model.state_dict(),
            'val_acc':va,'val_f1':vf1,'idx2clase':idx2clase,
            'num_classes':NUM_CLASES,'model_name':MODEL_NAME,
            'img_size':IMG_SIZE,'mean':MEAN,'std':STD,
            'dataset':'PlantVillage',
        }, MODEL_SAVE)
        print(f'         💾 Checkpoint guardado — val_acc={va:.4f}  val_f1={vf1:.4f}')
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            print(f'\n⛔ Early stopping en epoch {epoch}')
            break

pd.DataFrame(history).to_csv(BASE_PATH/'data'/'historial.csv', index=False)
print(f'\n🏆 Entrenamiento finalizado — Mejor val_acc: {best_acc:.4f} ({best_acc*100:.2f}%)')


---
## Celda 9 — Curvas de entrenamiento

In [ ]:
hist_df = pd.DataFrame(history)
best_ep = hist_df.loc[hist_df['val_acc'].idxmax(), 'epoch']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'PlantVillage · {MODEL_NAME} · {NUM_CLASES} clases', fontsize=12, y=1.02)
for ax, y1, y2, tit in [
    (axes[0],'train_loss','val_loss','Loss'),
    (axes[1],'train_acc','val_acc','Accuracy'),
]:
    ax.plot(hist_df['epoch'], hist_df[y1], label='Train', color='#3498db', lw=2)
    ax.plot(hist_df['epoch'], hist_df[y2], label='Val',   color='#e74c3c', lw=2, ls='--')
    ax.set_title(tit, fontweight='bold'); ax.set_xlabel('Época')
    ax.legend(); ax.grid(alpha=0.3)
axes[1].axvline(x=best_ep, color='#27ae60', ls='--', alpha=0.8, label=f'Mejor ({int(best_ep)})')
axes[1].legend()
axes[2].plot(hist_df['epoch'], hist_df['val_f1'], color='#9b59b6', lw=2)
axes[2].fill_between(hist_df['epoch'], hist_df['val_f1'], alpha=0.15, color='#9b59b6')
axes[2].set_title('F1 Macro (Val)', fontweight='bold'); axes[2].set_xlabel('Época'); axes[2].grid(alpha=0.3)
plt.tight_layout()
out = BASE_PATH/'data'/'curvas.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
print(f'📁 curvas.png guardado → {out}')


---
## Celda 10 — Evaluación en el conjunto de prueba

Con PlantVillage es normal ver accuracy >90% porque las diferencias visuales entre enfermedades son muy marcadas.
**Atención:** revisa el recall de las clases minoritarias (las que tienen pocas imágenes), que pueden tener métricas más bajas.

In [ ]:
ckpt = torch.load(MODEL_SAVE, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'✅ Mejor modelo cargado — epoch {ckpt["epoch"]} | val_acc={ckpt["val_acc"]:.4f}')

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        logits = model(imgs.to(DEVICE))
        probs  = F.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds); all_labels.extend(labels.numpy()); all_probs.extend(probs)

nombres_clases = [idx2clase[str(i)] for i in range(NUM_CLASES)]
test_acc = (np.array(all_preds) == np.array(all_labels)).mean()
print(f'\n🎯 Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print('\n📊 Reporte de clasificación:')
reporte = classification_report(all_labels, all_preds,
                                 target_names=nombres_clases, digits=3, output_dict=True)
print(classification_report(all_labels, all_preds, target_names=nombres_clases, digits=3))
pd.DataFrame(reporte).T.to_csv(BASE_PATH/'data'/'reporte.csv')
print('📁 reporte.csv guardado')


In [ ]:
# Matrices de confusión
# Con 38 clases la anotación de cada celda se desactiva (muy pequeña).
# La matriz normalizada muestra qué tan bien detecta cada clase (recall).
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:,np.newaxis]
anotar  = False   # 38 clases → sin anotación numérica (demasiado pequeño)

for fname, data, fmt, cmap, titulo in [
    ('confusion_counts.png', cm,      'd',   'Blues',  'Conteos absolutos'),
    ('confusion_norm.png',   cm_norm, '.2f', 'YlOrRd', 'Normalizada — Recall por clase'),
]:
    ts = 6  # tamaño de fuente para etiquetas de 38 clases
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(data, ax=ax, cmap=cmap, annot=anotar, fmt=fmt,
                xticklabels=nombres_clases, yticklabels=nombres_clases)
    ax.set_title(f'Matriz de Confusión PlantVillage — {titulo}',
                 fontweight='bold', fontsize=12)
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
    ax.tick_params(axis='x', rotation=45, labelsize=ts)
    ax.tick_params(axis='y', rotation=0,  labelsize=ts)
    plt.tight_layout()
    out = BASE_PATH/'data'/fname
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
    print(f'📁 {fname} guardado')


In [ ]:
# Métricas por clase
# Con 38 clases, las etiquetas del eje X van rotadas.
# Las líneas punteadas marcan los umbrales de aceptable (0.5) y bueno (0.8).
clases_plot = [c for c in nombres_clases if c in reporte]
prec_v = [reporte[c]['precision'] for c in clases_plot]
rec_v  = [reporte[c]['recall']    for c in clases_plot]
f1_v   = [reporte[c]['f1-score']  for c in clases_plot]
x = np.arange(len(clases_plot)); w = 0.25
fig, ax = plt.subplots(figsize=(18, 5))
ax.bar(x-w, prec_v, w, label='Precision', color='#3498db', alpha=0.85)
ax.bar(x,   rec_v,  w, label='Recall',    color='#27ae60', alpha=0.85)
ax.bar(x+w, f1_v,   w, label='F1-Score',  color='#9b59b6', alpha=0.85)
ax.axhline(y=0.5, color='red',   ls='--', alpha=0.4, lw=1)
ax.axhline(y=0.8, color='green', ls='--', alpha=0.4, lw=1)
ax.set_xticks(x)
ax.set_xticklabels(clases_plot, rotation=45, ha='right', fontsize=6)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('Métricas por clase — PlantVillage (38 clases)', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out = BASE_PATH/'data'/'metricas_clase.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
print(f'📁 metricas_clase.png guardado → {out}')


---
## Celda 11 — Visualización de predicciones

In [ ]:
model.eval()
idxs = random.sample(range(len(test_ds)), min(16, len(test_ds)))
fig, axes = plt.subplots(2, 8, figsize=(22, 7))
fig.suptitle('PlantVillage — Predicciones del test set (verde=correcto · rojo=error)',
             fontsize=12, fontweight='bold')
for ax, idx in zip(axes.ravel(), idxs):
    path, lbl_real = test_ds.samples[idx]
    try:
        img_np = np.array(Image.open(path).convert('RGB'))
        tensor = val_aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = F.softmax(model(tensor), dim=1)[0]
            pred  = probs.argmax().item()
            conf  = probs[pred].item() * 100
        ax.imshow(img_np)
        ok    = pred == lbl_real
        color = '#27ae60' if ok else '#e74c3c'
        # Mostrar solo la parte de la enfermedad (después de ___)
        nombre = idx2clase[str(pred)].split('___')[-1][:14]
        ax.set_title(f'{"✅" if ok else "❌"}\n{nombre}\n{conf:.0f}%',
                     fontsize=6.5, color=color)
    except Exception: pass
    ax.axis('off')
plt.tight_layout()
out = BASE_PATH/'data'/'predicciones_grid.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
print(f'📁 predicciones_grid.png guardado → {out}')


In [ ]:
# Probar con tu propia imagen de hoja
MI_IMAGEN = r'C:\ruta\a\tu\hoja.jpg'

img_path = Path(MI_IMAGEN)
if not img_path.exists():
    print(f'⚠️  Imagen no encontrada: {img_path}')
    print('   Cambia MI_IMAGEN a la ruta de una foto de hoja de tu PC.')
    print('   Funciona mejor con hojas de las 14 plantas del dataset.')
else:
    img_np = np.array(Image.open(img_path).convert('RGB'))
    tensor = val_aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs        = F.softmax(model(tensor), dim=1)[0].cpu()
        top5p, top5i = torch.topk(probs, min(5, NUM_CLASES))

    nombre_pred = idx2clase[str(top5i[0].item())]
    # Separar planta y enfermedad para mostrar más claro
    partes = nombre_pred.split('___')
    titulo = f'{partes[0]} — {partes[1]}' if len(partes)>1 else nombre_pred

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Tu hoja — Clasificada como: {titulo}', fontsize=12, fontweight='bold')
    axes[0].imshow(img_np); axes[0].set_title(img_path.name, fontsize=10); axes[0].axis('off')

    top_n = [idx2clase[str(i.item())].replace('___',' — ')[:28] for i in top5i]
    top_p = [p.item()*100 for p in top5p]
    axes[1].barh(top_n[::-1], top_p[::-1], color='#27ae60', alpha=0.85, edgecolor='white')
    for i, prob in enumerate(top_p[::-1]):
        axes[1].text(prob+0.5, i, f'{prob:.1f}%', va='center', fontsize=9)
    axes[1].set_xlim(0, 110); axes[1].set_xlabel('Probabilidad (%)')
    axes[1].set_title('Top-5 predicciones'); axes[1].grid(axis='x', alpha=0.3)
    plt.tight_layout()
    out = BASE_PATH/'data'/'mi_prediccion.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
    print('🎯 Resultado:')
    for i,(p,idx) in enumerate(zip(top5p,top5i)):
        n = idx2clase[str(idx.item())].replace('___',' — ')
        print(f'   Top-{i+1}: {n:<40} {p.item()*100:.2f}%')
    print(f'\n📁 mi_prediccion.png guardado → {out}')


---
## Celda 12 — Exportar modelo para producción

In [ ]:
ckpt = torch.load(MODEL_SAVE, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict']); model.eval()

prod_path = BASE_PATH/'models'/'modelo_produccion.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'idx2clase': idx2clase, 'num_classes': NUM_CLASES,
    'model_name': MODEL_NAME, 'img_size': IMG_SIZE,
    'mean': MEAN, 'std': STD,
    'val_acc': ckpt['val_acc'], 'val_f1': ckpt['val_f1'],
    'best_epoch': ckpt['epoch'], 'dataset': 'PlantVillage',
    'clases': clases_train,
}, prod_path)
print(f'✅ modelo_produccion.pth — {prod_path.stat().st_size/1e6:.1f} MB')
with open(BASE_PATH/'data'/'idx2clase.json','w',encoding='utf-8') as f:
    json.dump(idx2clase, f, ensure_ascii=False, indent=2)
print('✅ idx2clase.json guardado')


---
## Celda 13 — Resumen final

In [ ]:
ckpt = torch.load(MODEL_SAVE, map_location='cpu')
print('=' * 64)
print('  🌿 ACTIVIDAD FINAL — CLASIFICACIÓN DE ENFERMEDADES EN PLANTAS')
print('  Dataset: PlantVillage (38 clases)')
print('=' * 64)
print(f'  Modelo        : {MODEL_NAME}')
print(f'  PC débil      : {PC_DEBIL}')
print(f'  Dispositivo   : {DEVICE}')
print(f'  Clases        : {NUM_CLASES}')
print(f'  Imágenes      : train={len(train_ds):,} | val={len(val_ds):,} | test={len(test_ds):,}')
print(f'  Mejor época   : {ckpt["epoch"]}')
print(f'  Val Accuracy  : {ckpt["val_acc"]*100:.2f}%')
print(f'  Val F1 Macro  : {ckpt["val_f1"]*100:.2f}%')
print(f'  Test Accuracy : {test_acc*100:.2f}%')
print('=' * 64)
archivos = [
    BASE_PATH/'models'/'modelo_best.pth',
    BASE_PATH/'models'/'modelo_produccion.pth',
    BASE_PATH/'data'/'idx2clase.json',
    BASE_PATH/'data'/'historial.csv',
    BASE_PATH/'data'/'reporte.csv',
    BASE_PATH/'data'/'curvas.png',
    BASE_PATH/'data'/'distribucion.png',
    BASE_PATH/'data'/'muestras_dataset.png',
    BASE_PATH/'data'/'confusion_counts.png',
    BASE_PATH/'data'/'confusion_norm.png',
    BASE_PATH/'data'/'metricas_clase.png',
    BASE_PATH/'data'/'predicciones_grid.png',
]
print('  Archivos generados:')
for r in archivos:
    ok   = r.exists()
    size = f'{r.stat().st_size/1e6:.1f} MB' if ok else 'no encontrado'
    print(f'  {"✅" if ok else "⚠️"} {r.name:<38} {size}')
print('=' * 64)
